In [1]:
!pip install medpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for medpy: filename=MedPy-0.5.2-py3-none-any.whl size=224710 sha256=fb41ba46f213e3b50a3d3ff9cd34d16d1adb0aa7644a36568894d486c9304e8f
  Stored in directory: /root/.cache/pip/wheels/d4/33/ed/aaac5a347fb8d41679ca515b8f5c49dfdf49be15bdbb9a905d
Successfully built medpy


In [2]:
# data_setup.py

import os
import numpy as np
import nibabel as nib
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from tqdm import tqdm 
from medpy.metric import binary # For Dice and Hausdorff metrics


# --- 1. Helper Function to Collect Patient Information ---
def collect_patient_info_from_root(data_root_path, grade_subfolders=True):
    """
    Scans a given BraTS root directory to collect patient information.

    Args:
        data_root_path (str): The root path to the BraTS data.
        grade_subfolders (bool): If True, expects 'HGG' and 'LGG' subfolders.
                                 If False, expects patient folders directly under data_root_path.

    Returns:
        list: A list of dictionaries, where each dictionary contains
              'id', 'path', and 'grade' for a patient.
    """
    patients_info = []

    if grade_subfolders:
        hgg_path = os.path.join(data_root_path, 'HGG')
        lgg_path = os.path.join(data_root_path, 'LGG')

        # Process HGG patients
        if os.path.exists(hgg_path):
            for patient_folder_name in os.listdir(hgg_path):
                patient_full_path = os.path.join(hgg_path, patient_folder_name)
                if os.path.isdir(patient_full_path):
                    patients_info.append({
                        'id': patient_folder_name,
                        'path': patient_full_path,
                        'grade': 'HGG'
                    })
        # else: # Optional: Uncomment for warnings if paths are missing
        #     print(f"Warning: HGG path not found in {data_root_path}: {hgg_path}")

        # Process LGG patients
        if os.path.exists(lgg_path):
            for patient_folder_name in os.listdir(lgg_path):
                patient_full_path = os.path.join(lgg_path, patient_folder_name)
                if os.path.isdir(patient_full_path):
                    patients_info.append({
                        'id': patient_folder_name,
                        'path': patient_full_path,
                        'grade': 'LGG'
                    })
        # else: # Optional: Uncomment for warnings if paths are missing
        #     print(f"Warning: LGG path not found in {data_root_path}: {lgg_path}")
    else: # No HGG/LGG subfolders, patient folders are direct children
        # print(f"Attempting to find patient folders directly under: {data_root_path}") # Optional print
        if os.path.exists(data_root_path):
            for patient_folder_name in os.listdir(data_root_path):
                patient_full_path = os.path.join(data_root_path, patient_folder_name)
                # Ensure it's a directory and looks like a patient folder (e.g., starts with BraTS)
                if os.path.isdir(patient_full_path) and patient_folder_name.startswith('BraTS'):
                    patients_info.append({
                        'id': patient_folder_name,
                        'path': patient_full_path,
                        'grade': 'Unknown' # Placeholder grade for validation/test set
                    })
        # else: # Optional: Uncomment for warnings if paths are missing
        #     print(f"Error: Data root path not found: {data_root_path}")

    return patients_info


# --- 2. BraTSDataset Class (YOUR ORIGINAL VERSION) ---
# This version implies that segmentation masks MUST be present for all patients.
class BraTSDataset(Dataset):
    def __init__(self, patients, transform=None, mode='train', slice_selection_method='all'):
        self.patients = patients
        self.transform = transform
        self.mode = mode
        self.slice_selection_method = slice_selection_method

        self.slices = []

        for patient in self.patients:
            patient_id = patient['id']
            patient_path = patient['path']
            grade = patient['grade']

            t1_path = os.path.join(patient_path, f"{patient_id}_t1.nii")
            t1ce_path = os.path.join(patient_path, f"{patient_id}_t1ce.nii")
            t2_path = os.path.join(patient_path, f"{patient_id}_t2.nii")
            flair_path = os.path.join(patient_path, f"{patient_id}_flair.nii")
            seg_path = os.path.join(patient_path, f"{patient_id}_seg.nii")

            if not all(os.path.exists(p) for p in [t1_path, t1ce_path, t2_path, flair_path, seg_path]):
                t1_path = os.path.join(patient_path, f"{patient_id}_T1.nii.gz")
                t1ce_path = os.path.join(patient_path, f"{patient_id}_T1CE.nii.gz")
                t2_path = os.path.join(patient_path, f"{patient_id}_T2.nii.gz")
                flair_path = os.path.join(patient_path, f"{patient_id}_FLAIR.nii.gz")
                seg_path = os.path.join(patient_path, f"{patient_id}_seg.nii.gz")

            if not all(os.path.exists(p) for p in [t1_path, t1ce_path, t2_path, flair_path, seg_path]):
                print(f"Skipping {patient_id}: Missing required files")
                continue

            seg_img = nib.load(seg_path).get_fdata() # This line implicitly expects seg_path to exist

            if slice_selection_method == 'all':
                for slice_idx in range(seg_img.shape[2]):
                    # Include slices within the central 80% range
                    if seg_img.shape[2] * 0.1 <= slice_idx <= seg_img.shape[2] * 0.9:
                        self.slices.append({
                            'patient': patient_id,
                            'grade': grade,
                            'slice_idx': slice_idx,
                            't1_path': t1_path,
                            't1ce_path': t1ce_path,
                            't2_path': t2_path,
                            'flair_path': flair_path,
                            'seg_path': seg_path
                        })
            elif slice_selection_method == 'tumor_only':
                for slice_idx in range(seg_img.shape[2]):
                    if np.any(seg_img[:, :, slice_idx] > 0):
                        self.slices.append({
                            'patient': patient_id,
                            'grade': grade,
                            'slice_idx': slice_idx,
                            't1_path': t1_path,
                            't1ce_path': t1ce_path,
                            't2_path': t2_path,
                            'flair_path': flair_path,
                            'seg_path': seg_path
                        })
            else:
                raise ValueError(f"Invalid slice selection method: {slice_selection_method}")

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        slice_data = self.slices[idx]
        patient = slice_data['patient']
        slice_idx = slice_data['slice_idx']

        t1 = nib.load(slice_data['t1_path']).get_fdata()[:, :, slice_idx]
        t1ce = nib.load(slice_data['t1ce_path']).get_fdata()[:, :, slice_idx]
        t2 = nib.load(slice_data['t2_path']).get_fdata()[:, :, slice_idx]
        flair = nib.load(slice_data['flair_path']).get_fdata()[:, :, slice_idx]
        seg = nib.load(slice_data['seg_path']).get_fdata()[:, :, slice_idx]

        # Preprocess
        t1 = self._preprocess(t1)
        t1ce = self._preprocess(t1ce)
        t2 = self._preprocess(t2)
        flair = self._preprocess(flair)

        image = np.stack([t1, t1ce, t2, flair], axis=0).astype(np.float32)

        mask = np.zeros((4, *seg.shape), dtype=np.float32)
        mask[0, seg == 0] = 1  # Background
        mask[1, seg == 1] = 1  # NCR/NET
        mask[2, seg == 2] = 1  # ED
        mask[3, seg == 4] = 1  # ET

        if self.transform:
            # Albumentations expects (H, W, C) for image and mask
            transformed = self.transform(image=image.transpose(1, 2, 0), mask=mask.transpose(1, 2, 0))
            image = transformed['image'].transpose(2, 0, 1) # Back to (C, H, W)
            mask = transformed['mask'].transpose(2, 0, 1)   # Back to (C, H, W)

        return {
            'image': torch.from_numpy(image),
            'mask': torch.from_numpy(mask),
            'patient': patient,
            'slice': slice_idx
        }

    def _preprocess(self, img):
        mean = np.mean(img)
        std = np.std(img)
        if std > 0:
            img = (img - mean) / std

        img = np.clip(img, -5, 5)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        return img

# --- 3. Albumentations Transforms ---
def get_train_transforms(size=240):
    return A.Compose([
        A.Resize(size, size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(
            translate_percent={'x': (-0.0625, 0.0625), 'y': (-0.0625, 0.0625)},
            scale=(0.9, 1.1),
            rotate=(-15, 15),
            p=0.5
        ),
        A.OneOf([
            A.GridDistortion(num_steps=5, distort_limit=0.05, p=1.0),
            A.ElasticTransform(alpha=1, sigma=50, p=1.0)
        ], p=0.25),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
        A.GaussNoise(std_range=(0.04, 0.2), mean_range=(0.0, 0.0), p=0.25),
    ])

def get_val_transforms(size=240):
    return A.Compose([
        A.Resize(size, size),
    ])

# --- 4. Segmentation Metrics ---
def dice_coefficient(y_true, y_pred):
    """
    Calculates the Dice coefficient.
    Args:
        y_true (np.array): Ground truth binary mask.
        y_pred (np.array): Predicted binary mask.
    Returns:
        float: Dice coefficient. Returns 1.0 if both masks are empty, 0.0 otherwise if one is empty.
    """
    if y_true.sum() == 0 and y_pred.sum() == 0:
        return 1.0
    if y_true.sum() == 0 or y_pred.sum() == 0:
        return 0.0
    return binary.dc(y_pred.astype(bool), y_true.astype(bool))

def hausdorff_distance_95(y_true, y_pred):
    """
    Calculates the 95th percentile Hausdorff Distance.
    Args:
        y_true (np.array): Ground truth binary mask.
        y_pred (np.array): Predicted binary mask.
    Returns:
        float: 95th percentile Hausdorff Distance. Returns 0.0 if both masks are empty,
               np.inf if one is empty and the other is not.
    """
    if y_true.sum() == 0 and y_pred.sum() == 0:
        return 0.0
    if y_true.sum() == 0 or y_pred.sum() == 0:
        return np.inf
    return binary.hd95(y_pred.astype(bool), y_true.astype(bool))


# --- Main function to get DataLoaders ---
def get_brats_dataloaders(
    train_data_root='/kaggle/input/miccaibrats2019/MICCAI_BraTS_2019_Data_Training/MICCAI_BraTS_2019_Data_Training',
    train_ratio=0.70,
    val_ratio=0.15,
    local_test_ratio=0.15,
    random_state=42,
    batch_size=4,
    num_workers=os.cpu_count()
):
    """
    Collects patient information and divides it into training, validation,
    and a local test set, returning corresponding PyTorch DataLoaders.

    Args:
        train_data_root (str): Path to the BraTS 2019 training data.
        train_ratio (float): Proportion of data for the training set.
        val_ratio (float): Proportion of data for the validation set.
        local_test_ratio (float): Proportion of data for the local test set.
        random_state (int): Seed for reproducible splits.
        batch_size (int): Batch size for DataLoaders.
        num_workers (int): Number of worker processes for DataLoaders.

    Returns:
        tuple: (train_loader, val_loader, local_test_loader)
    """
    if not np.isclose(train_ratio + val_ratio + local_test_ratio, 1.0):
        raise ValueError("Train, Val, and Local Test ratios must sum to 1.0")

    print(f"Collecting patient information from: {train_data_root}")
    all_patients = collect_patient_info_from_root(train_data_root, grade_subfolders=True)
    print(f"Collected info for {len(all_patients)} total patients.")

    # Step 1: Separate out the local test set first
    train_val_patients, local_test_patients = train_test_split(
        all_patients,
        test_size=local_test_ratio,
        random_state=random_state
    )
    print(f"Split: {len(train_val_patients)} patients for train/val, {len(local_test_patients)} patients for local test.")

    # Step 2: Divide the remaining into train and validation
    # Adjust validation ratio based on the new size of `train_val_patients`
    adjusted_val_ratio = val_ratio / (train_ratio + val_ratio)

    train_patients, val_patients = train_test_split(
        train_val_patients,
        test_size=adjusted_val_ratio,
        random_state=random_state
    )
    print(f"Further split: {len(train_patients)} patients for training, {len(val_patients)} patients for validation.")

    # Initialize Datasets
    train_dataset = BraTSDataset(
        patients=train_patients,
        transform=get_train_transforms(),
        mode='train',
        slice_selection_method='tumor_only'
    )
    val_dataset = BraTSDataset(
        patients=val_patients,
        transform=get_val_transforms(),
        mode='val',
        slice_selection_method='tumor_only'
    )
    local_test_dataset = BraTSDataset(
        patients=local_test_patients,
        transform=get_val_transforms(),
        mode='test',
        slice_selection_method='tumor_only'
    )

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    local_test_loader = DataLoader(local_test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    print(f"\n--- DataLoaders Summary ---")
    print(f"Train Loader: {len(train_dataset)} slices, Batch Size: {train_loader.batch_size}")
    print(f"Validation Loader: {len(val_dataset)} slices, Batch Size: {val_loader.batch_size}")
    print(f"Local Test Loader: {len(local_test_dataset)} slices, Batch Size: {local_test_loader.batch_size}")

    return train_loader, val_loader, local_test_loader

#  if you run data_setup.py directly 
if __name__ == '__main__':
    train_loader, val_loader, local_test_loader = get_brats_dataloaders(
        train_data_root='/kaggle/input/miccaibrats2019/MICCAI_BraTS_2019_Data_Training/MICCAI_BraTS_2019_Data_Training',
        batch_size=8 # Example batch size
    )
    print("\nDataLoaders created successfully for local testing!")
    # You can further inspect loaders here, e.g., next(iter(train_loader))

/usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Collected info for 335 total patients.
Split: 284 patients for train/val, 51 patients for local test.
Further split: 233 patients for training, 51 patients for validation.

--- DataLoaders Summary ---
Train Loader: 15328 slices, Batch Size: 8
Validation Loader: 3367 slices, Batch Size: 8
Local Test Loader: 3455 slices, Batch Size: 8

DataLoaders created successfully for local testing!


In [3]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import os

class DAE(nn.Module):
    def __init__(self, image_shape, structure, v_noise=0.1, activation=nn.ReLU, reg_strength=1e-4):
        super(DAE, self).__init__()
        
        self.image_shape = image_shape
        self.structure = structure
        self.v_noise = v_noise
        self.reg_strength = reg_strength
        
        # Handle activation parameter - convert string to actual activation class if needed
        if isinstance(activation, str):
            activation_map = {
                'relu': nn.ReLU,
                'leaky_relu': nn.LeakyReLU,
                'tanh': nn.Tanh,
                'sigmoid': nn.Sigmoid,
                'elu': nn.ELU,
                'gelu': nn.GELU
            }
            activation_fn = activation_map.get(activation.lower(), nn.ReLU)
        else:
            activation_fn = activation
        
        # Extract channel information and special operations
        channels = []
        operations = []
        
        for item in structure:
            if isinstance(item, int):
                channels.append(item)
                operations.append('conv')
            elif item == "max":
                operations.append('maxpool')
            elif item == "linear_bottleneck":
                operations.append('linear_bottleneck')
        
        # Build encoder
        self.encoder_layers = nn.ModuleList()
        
        in_channels = image_shape[0]  # Initial input channels (4 for BraTS)
        current_size = image_shape[1]  # Assuming square images (240x240)
        
        channel_idx = 0
        for i, op in enumerate(operations):
            if op == 'conv':
                out_channels = channels[channel_idx]
                self.encoder_layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
                self.encoder_layers.append(activation_fn(inplace=True))
                in_channels = out_channels
                channel_idx += 1
                
            elif op == 'maxpool':
                self.encoder_layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
                current_size = current_size // 2
                
            elif op == 'linear_bottleneck':
                # Flatten and create bottleneck
                flattened_size = in_channels * current_size * current_size
                bottleneck_size = channels[channel_idx]
                
                self.encoder_layers.append(nn.Flatten())
                self.encoder_layers.append(nn.Linear(flattened_size, bottleneck_size))
                self.encoder_layers.append(activation_fn(inplace=True))
                
                # Store info for decoder
                self.bottleneck_size = bottleneck_size
                self.pre_flatten_channels = in_channels
                self.pre_flatten_size = current_size
                channel_idx += 1
                break
        
        # Build decoder (reverse of encoder)
        self.decoder_layers = nn.ModuleList()
        
        # Handle bottleneck reconstruction
        if 'linear_bottleneck' in [op for op in operations]:
            # Reverse the linear bottleneck
            reconstruct_size = self.pre_flatten_channels * self.pre_flatten_size * self.pre_flatten_size
            self.decoder_layers.append(nn.Linear(self.bottleneck_size, reconstruct_size))
            self.decoder_layers.append(activation_fn(inplace=True))
            
            # Reshape back to feature maps
            # This will be handled in forward pass
            current_channels = self.pre_flatten_channels
            current_size = self.pre_flatten_size
        else:
            current_channels = channels[-1]
        
        # Reverse the convolutional layers
        reversed_channels = channels[:-1] if 'linear_bottleneck' in operations else channels[:-1]
        reversed_channels.reverse()
        reversed_channels.append(image_shape[0])  # Back to original input channels
        
        # Count maxpool operations to know how many upsample layers we need
        num_maxpools = operations.count('maxpool')
        
        # Add transposed convolutions and upsampling
        for i in range(len(reversed_channels)):
            if i < num_maxpools:
                # Add upsampling first
                self.decoder_layers.append(nn.Upsample(scale_factor=2, mode='nearest'))
            
            out_channels = reversed_channels[i]
            self.decoder_layers.append(nn.ConvTranspose2d(current_channels, out_channels, kernel_size=3, padding=1))
            
            # Don't add activation after the final layer
            if i < len(reversed_channels) - 1:
                self.decoder_layers.append(activation_fn(inplace=True))
            
            current_channels = out_channels
    
    def add_noise(self, x):
        if self.training and self.v_noise > 0:
            noise = torch.randn_like(x) * self.v_noise
            return x + noise
        return x
    
    def forward(self, x):
        # Add noise for denoising
        x = self.add_noise(x)
        
        # Encoder
        for layer in self.encoder_layers:
            x = layer(x)
        
        # Decoder
        need_reshape = False
        reshape_channels = None
        reshape_size = None
        
        for i, layer in enumerate(self.decoder_layers):
            if isinstance(layer, nn.Linear) and i == 0:
                # Store info for reshaping after linear layer
                need_reshape = True
                reshape_channels = self.pre_flatten_channels
                reshape_size = self.pre_flatten_size
                
            x = layer(x)
            
            # Reshape after first linear layer in decoder
            if need_reshape and isinstance(layer, nn.Linear):
                x = x.view(-1, reshape_channels, reshape_size, reshape_size)
                need_reshape = False
        
        return x



In [4]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import os
import wandb

# --- Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 4
epochs = 8
reg_strength = 1e-9
activation = "relu"

# --- Input shape for 2D slices: (C, H, W) ---
input_shape = (4, 240, 240)

# --- Architecture structures for 2D Conv DAE ---
# Very lightweight for 2D slice-based BraTS
structure_AE_I = [
    32,        # Conv2d: 4 → 16
    "max",     # → 120x120
    64,        # Conv2d
    "max",
    128,
    "max",
    "linear_bottleneck",
    2048
            # Bottleneck features
]



structure_AE_II = [
    64,
    "max",      # -> 120x120
    128,
    "max",      # -> 60x60
    256,
    "max",      # -> 30x30
    512        
]


def train_autoencoder(model, train_loader, val_loader, archive_name):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=reg_strength)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')

    print(f"\n--- Starting training for {archive_name} ---")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, batch_data in enumerate(train_loader):
            imgs = batch_data['image']  # Expect shape: [B, 4, 240, 240]

            if epoch == 0 and batch_idx == 0:
                print(f"Input batch shape: {imgs.shape}")  # [B, 4, 240, 240]
                assert imgs.dim() == 4, f"Expected 4D tensor (B,C,H,W), got {imgs.shape}"

            noisy_imgs = imgs + model.v_noise * torch.randn_like(imgs)
            noisy_imgs = torch.clamp(noisy_imgs, 0.0, 1.0)

            noisy_imgs, imgs = noisy_imgs.to(device), imgs.to(device)

            optimizer.zero_grad()
            outputs = model(noisy_imgs)
            loss = criterion(outputs, imgs)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            if (batch_idx + 1) % 10 == 0 or batch_idx == len(train_loader) - 1:
                print(f"Epoch [{epoch + 1}/{epochs}], Batch [{batch_idx + 1}/{len(train_loader)}], Loss: {loss.item():.4f}")

        avg_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_data in val_loader:
                imgs = batch_data['image']
                noisy_imgs = imgs + model.v_noise * torch.randn_like(imgs)
                noisy_imgs = torch.clamp(noisy_imgs, 0.0, 1.0)
                noisy_imgs, imgs = noisy_imgs.to(device), imgs.to(device)
                outputs = model(noisy_imgs)
                loss = criterion(outputs, imgs)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch [{epoch + 1}/{epochs}] - Train Loss: {avg_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            os.makedirs("./defensive_models/", exist_ok=True)
            torch.save(model.state_dict(), f"./defensive_models/{archive_name}_best.pth")
            print(f"  --> Best model for {archive_name} saved with Val Loss: {best_val_loss:.4f}")

    torch.save(model.state_dict(), f"./defensive_models/{archive_name}_final.pth")
    print(f"Final model for {archive_name} saved.")

if __name__ == "__main__":
    print("Loading BraTS data...")
    train_loader, val_loader, local_test_loader = get_brats_dataloaders(
        train_data_root='/kaggle/input/miccaibrats2019/MICCAI_BraTS_2019_Data_Training/MICCAI_BraTS_2019_Data_Training',
        batch_size=batch_size,
        num_workers=min(4, os.cpu_count())
          # <- Optional if your loader supports this switch
    )
    print("BraTS DataLoaders ready.")

    # from DAE_model import DAE  # Your updated 2D DAE model

    """print(f"\nCreating DAE Model I with input shape: {input_shape}")
    AE_I = DAE(
        image_shape=input_shape,
        structure=structure_AE_I,
        v_noise=0.1,
        activation=activation,
        reg_strength=reg_strength
    )
    train_autoencoder(AE_I, train_loader, val_loader, "BraTS_DAE2D_I")"""

    print(f"\nCreating DAE Model II with input shape: {input_shape}")
    AE_II = DAE(
        image_shape=input_shape,
        structure=structure_AE_II,
        v_noise=0.1,
        activation=activation,
        reg_strength=reg_strength
    )
    train_autoencoder(AE_II, train_loader, val_loader, "BraTS_DAE2D_II")

    print("\nTraining complete for both BraTS 2D Denoising Autoencoders!")
    print(f"Local Test Loader has {len(local_test_loader.dataset)} samples.")



Loading BraTS data...
Collected info for 335 total patients.
Split: 284 patients for train/val, 51 patients for local test.
Further split: 233 patients for training, 51 patients for validation.

--- DataLoaders Summary ---
Train Loader: 15328 slices, Batch Size: 4
Validation Loader: 3367 slices, Batch Size: 4
Local Test Loader: 3455 slices, Batch Size: 4
BraTS DataLoaders ready.

Creating DAE Model II with input shape: (4, 240, 240)

--- Starting training for BraTS_DAE2D_II ---
Input batch shape: torch.Size([4, 4, 240, 240])
Epoch [1/8], Batch [10/3832], Loss: 0.0458
Epoch [1/8], Batch [20/3832], Loss: 0.0175
Epoch [1/8], Batch [30/3832], Loss: 0.0169
Epoch [1/8], Batch [40/3832], Loss: 0.0097
Epoch [1/8], Batch [50/3832], Loss: 0.0110
Epoch [1/8], Batch [60/3832], Loss: 0.0081
Epoch [1/8], Batch [70/3832], Loss: 0.0137
Epoch [1/8], Batch [80/3832], Loss: 0.0063
Epoch [1/8], Batch [90/3832], Loss: 0.0051
Epoch [1/8], Batch [100/3832], Loss: 0.0039
Epoch [1/8], Batch [110/3832], Loss: 0